In [1]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """Training configuration"""
    # GCS Settings
    GCS_BUCKET = "processed_data-iyed"
    GCS_PREFIX = "processed_regression/"

    # Data files
    SPATIAL_TRAIN = "X_spatial_train.h5"
    SPATIAL_TEST = "X_spatial_test.h5"
    SPECTRAL_TRAIN = "X_spectral_train.npy"
    SPECTRAL_TEST = "X_spectral_test.npy"
    LABELS_TRAIN = "Y_train.npy"
    LABELS_TEST = "Y_test.npy"

    # Model parameters
    SPATIAL_SHAPE = (11, 11, 141)
    SPECTRAL_SHAPE = (141,)
    OUTPUT_DIM = 1

    # Training parameters
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5

    # CNN parameters
    CNN_FILTERS = [32, 64, 128, 256]
    CNN_DROPOUT = 0.3

    # Transformer parameters
    TRANSFORMER_HEADS = 8
    TRANSFORMER_LAYERS = 3
    TRANSFORMER_DIM = 128
    TRANSFORMER_FF_DIM = 256

    # Fusion parameters
    FUSION_DIM = 256

    # Classifier parameters
    CLASSIFIER_DIMS = [512, 256, 128]
    CLASSIFIER_DROPOUT = 0.4

    # Callbacks
    EARLY_STOPPING_PATIENCE = 15
    REDUCE_LR_PATIENCE = 8

    # Paths
    CHECKPOINT_DIR = "/content/drive/MyDrive/spectrofood_checkpoints"
    RESULTS_DIR = "results"

config = Config()

# Create directories

# ============================================================================
# GCS AUTHENTICATION & DATA DOWNLOAD
# ============================================================================
print("\n[1/10] Setting up GCS and downloading data...")

from google.colab import auth
auth.authenticate_user()

storage_client = storage.Client()
bucket = storage_client.bucket(config.GCS_BUCKET)

def download_from_gcs(gcs_path: str, local_path: str) -> None:
    """Download file from GCS to local storage"""
    blob = bucket.blob(gcs_path)
    blob.download_to_filename(local_path)
    size_mb = os.path.getsize(local_path) / (1024**2)
    print(f"  ✓ Downloaded {local_path} ({size_mb:.1f} MB)")

files_to_download = [
    # (config.GCS_PREFIX + config.SPATIAL_TRAIN, "data/" + config.SPATIAL_TRAIN),
    # (config.GCS_PREFIX + config.SPATIAL_TEST, "data/" + config.SPATIAL_TEST),
    # (config.GCS_PREFIX + config.SPECTRAL_TRAIN, "data/" + config.SPECTRAL_TRAIN),
    # (config.GCS_PREFIX + config.SPECTRAL_TEST, "data/" + config.SPECTRAL_TEST),
    (config.GCS_PREFIX + config.LABELS_TRAIN, "data/" + config.LABELS_TRAIN),
    (config.GCS_PREFIX + config.LABELS_TEST, "data/" + config.LABELS_TEST),
]

for gcs_path, local_path in files_to_download:
    download_from_gcs(gcs_path, local_path)



[1/10] Setting up GCS and downloading data...


MessageError: Error: credential propagation was unsuccessful

In [2]:
"""
🔒 REGRESSION-BASED PREPROCESSING PIPELINE
SpectroFood Apple Maturity Dataset
Predicting Continuous Dry Matter % (No Classification)
"""

import numpy as np
import pandas as pd
import h5py
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
import os
from tqdm import tqdm
import gc

print("=" * 80)
print("SpectroFood Apple Maturity Dataset - REGRESSION Preprocessing Pipeline")
print("=" * 80)

SpectroFood Apple Maturity Dataset - REGRESSION Preprocessing Pipeline


In [4]:
# ============================================================================
# STEP 1: LOAD MATLAB FILE
# ============================================================================
print("\n[1/8] Loading Apple.mat...")
mat_data = loadmat('/content/Apple.mat')

# Extract wavelengths
wavelengths = mat_data['Wavelengths'].flatten()
print(f"  ✓ Loaded {len(wavelengths)} wavelength bands")

# Extract all 240 apple hypercubes
apples = {}
for i in range(1, 241):
    var_name = f'A{i}'
    if var_name in mat_data:
        apples[i-1] = mat_data[var_name]
    else:
        raise ValueError(f"Missing apple {var_name} in MAT file")

print(f"  ✓ Loaded {len(apples)} apple hypercubes")


[1/8] Loading Apple.mat...
  ✓ Loaded 141 wavelength bands
  ✓ Loaded 240 apple hypercubes


In [5]:
# ============================================================================
# STEP 2: LOAD GROUND TRUTH CSV (DRY MATTER VALUES)
# ============================================================================
print("\n[2/8] Loading SpectroFood_dataset.csv...")
csv_data = pd.read_csv('/content/SpectroFood_dataset.csv')
csv_data.columns = csv_data.columns.str.strip()

# Build apple_id -> dry_matter mapping
apple_to_dry_matter = {}
for _, row in csv_data.iterrows():
    apple_name = str(row['Apple']).strip()

    if len(apple_name) < 2 or not apple_name[1:].isdigit():
        print(f"⚠️ Skipping invalid apple_name: {apple_name}")
        continue

    apple_id = int(apple_name[1:]) - 1
    apple_to_dry_matter[apple_id] = row['Dry matter']

print(f"  ✓ Loaded dry matter values for {len(apple_to_dry_matter)} apples")
print(f"  ✓ Dry matter range: {min(apple_to_dry_matter.values()):.4f} - {max(apple_to_dry_matter.values()):.4f}")

# Validate all apples have ground truth
missing_ids = set(range(240)) - set(apple_to_dry_matter.keys())
if missing_ids:
    raise ValueError(f"Missing dry matter values for apple IDs: {missing_ids}")



[2/8] Loading SpectroFood_dataset.csv...
  ✓ Loaded dry matter values for 240 apples
  ✓ Dry matter range: 0.1350 - 0.1743


In [6]:
# ============================================================================
# STEP 3: STATISTICS ON DRY MATTER DISTRIBUTION
# ============================================================================
print("\n[3/8] Analyzing dry matter distribution...")

all_dry_matters = np.array([apple_to_dry_matter[aid] for aid in range(240)])

print(f"\n  Dry matter statistics:")
print(f"    Min:  {all_dry_matters.min():.4f}")
print(f"    Max:  {all_dry_matters.max():.4f}")
print(f"    Mean: {all_dry_matters.mean():.4f}")
print(f"    Std:  {all_dry_matters.std():.4f}")
print(f"    Range: {all_dry_matters.max() - all_dry_matters.min():.4f}")

print(f"\n  Dry matter percentiles:")
for p in [10, 25, 50, 75, 90]:
    val = np.percentile(all_dry_matters, p)
    print(f"    {p:2d}%: {val:.4f}")



[3/8] Analyzing dry matter distribution...

  Dry matter statistics:
    Min:  0.1350
    Max:  0.1743
    Mean: 0.1552
    Std:  0.0084
    Range: 0.0394

  Dry matter percentiles:
    10%: 0.1432
    25%: 0.1493
    50%: 0.1555
    75%: 0.1615
    90%: 0.1663


In [7]:
# ============================================================================
# STEP 4: STRATIFIED TRAIN/TEST SPLIT AT APPLE LEVEL
# ============================================================================
print("\n[4/8] Stratified splitting of apples (80/20)...")

# Create quartile-based bins for stratification
# This ensures similar dry matter distributions in train/test
dry_matter_bins = pd.qcut(all_dry_matters, q=4, labels=False, duplicates='drop')

apple_ids = list(range(240))

train_apple_ids, test_apple_ids = train_test_split(
    apple_ids,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=dry_matter_bins  # Stratify by dry matter quartiles
)

print(f"  ✓ Train apples: {len(train_apple_ids)}")
print(f"  ✓ Test apples: {len(test_apple_ids)}")

# Check dry matter distributions in splits
train_dry_matters = [apple_to_dry_matter[aid] for aid in train_apple_ids]
test_dry_matters = [apple_to_dry_matter[aid] for aid in test_apple_ids]

print(f"\n  Train dry matter distribution:")
print(f"    Mean: {np.mean(train_dry_matters):.4f}, Std: {np.std(train_dry_matters):.4f}")
print(f"    Range: [{np.min(train_dry_matters):.4f}, {np.max(train_dry_matters):.4f}]")

print(f"\n  Test dry matter distribution:")
print(f"    Mean: {np.mean(test_dry_matters):.4f}, Std: {np.std(test_dry_matters):.4f}")
print(f"    Range: [{np.min(test_dry_matters):.4f}, {np.max(test_dry_matters):.4f}]")

# Critical check: no overlap
overlap = set(train_apple_ids) & set(test_apple_ids)
if overlap:
    raise ValueError(f"LEAKAGE DETECTED: Apples {overlap} appear in both splits!")
print(f"\n  ✓ Verified: Zero overlap between train and test apples")



[4/8] Stratified splitting of apples (80/20)...
  ✓ Train apples: 192
  ✓ Test apples: 48

  Train dry matter distribution:
    Mean: 0.1552, Std: 0.0086
    Range: [0.1350, 0.1743]

  Test dry matter distribution:
    Mean: 0.1552, Std: 0.0078
    Range: [0.1400, 0.1705]

  ✓ Verified: Zero overlap between train and test apples


In [8]:

# ============================================================================
# STEP 5: COUNT PATCHES
# ============================================================================
print("\n[5/8] Counting patches per split...")

PATCH_SIZE = 11
HALF_PATCH = PATCH_SIZE // 2

def count_patches_for_apple(apple_cube):
    h, w, bands = apple_cube.shape
    count = 0
    for y in range(HALF_PATCH, h - HALF_PATCH):
        for x in range(HALF_PATCH, w - HALF_PATCH):
            count += 1
    return count

# Count train patches
train_patch_count = 0
for apple_id in tqdm(train_apple_ids, desc="  Counting train patches"):
    train_patch_count += count_patches_for_apple(apples[apple_id])

# Count test patches
test_patch_count = 0
for apple_id in tqdm(test_apple_ids, desc="  Counting test patches"):
    test_patch_count += count_patches_for_apple(apples[apple_id])

print(f"  ✓ Train patches: {train_patch_count:,}")
print(f"  ✓ Test patches: {test_patch_count:,}")


[5/8] Counting patches per split...


  Counting test patches: 100%|██████████| 48/48 [00:00<00:00, 9104.03it/s]

  ✓ Train patches: 465,635
  ✓ Test patches: 110,844


In [ ]:

# ============================================================================
# STEP 6: EXTRACT AND SAVE TRAINING DATA
# ============================================================================
print("\n[6/8] Extracting TRAIN patches...")

h5_train = h5py.File('X_spatial_train.h5', 'w')
X_spatial_train_dataset = h5_train.create_dataset(
    'X_spatial',
    shape=(train_patch_count, PATCH_SIZE, PATCH_SIZE, 141),
    dtype=np.float32,
    chunks=(min(1000, train_patch_count), PATCH_SIZE, PATCH_SIZE, 141)
)

X_spectral_train = np.zeros((train_patch_count, 141), dtype=np.float32)
Y_train = np.zeros(train_patch_count, dtype=np.float32)  # Continuous targets!

patch_idx = 0
for apple_id in tqdm(train_apple_ids, desc="  Processing train apples"):
    cube = apples[apple_id].astype(np.float32)
    h, w, bands = cube.shape
    dry_matter = apple_to_dry_matter[apple_id]  # Continuous value

    for y in range(HALF_PATCH, h - HALF_PATCH):
        for x in range(HALF_PATCH, w - HALF_PATCH):
            spatial_patch = cube[y-HALF_PATCH:y+HALF_PATCH+1,
                                 x-HALF_PATCH:x+HALF_PATCH+1, :]
            spectral_vector = cube[y, x, :]

            X_spatial_train_dataset[patch_idx] = spatial_patch
            X_spectral_train[patch_idx] = spectral_vector
            Y_train[patch_idx] = dry_matter  # Store continuous dry matter

            patch_idx += 1

np.save('X_spectral_train.npy', X_spectral_train)
np.save('Y_train.npy', Y_train)

print(f"  ✓ Saved X_spatial_train.h5: shape {X_spatial_train_dataset.shape}")
print(f"  ✓ Saved X_spectral_train.npy: shape {X_spectral_train.shape}")
print(f"  ✓ Saved Y_train.npy: shape {Y_train.shape}")

del X_spectral_train, Y_train
gc.collect()

h5_train.close()


[6/8] Extracting TRAIN patches...


  Processing train apples: 100%|██████████| 192/192 [07:05<00:00,  2.22s/it]


  ✓ Saved X_spatial_train.h5: shape (465635, 11, 11, 141)
  ✓ Saved X_spectral_train.npy: shape (465635, 141)
  ✓ Saved Y_train.npy: shape (465635,)


In [12]:
# ============================================================================
# COMPUTE NORMALIZATION PARAMETERS (Do this ONCE before both steps)
# ============================================================================
csv_data = pd.read_csv('SpectroFood_dataset.csv')
all_dry_matters_vals = csv_data['Dry matter'].values
dm_min = all_dry_matters_vals.min()
dm_max = all_dry_matters_vals.max()

print(f"\n📊 Global Normalization Parameters:")
print(f"  Min: {dm_min:.6f}")
print(f"  Max: {dm_max:.6f}")
print(f"  Range: {dm_max - dm_min:.6f}")

# # ============================================================================
# # STEP 6: EXTRACT TRAIN DATA
# # ============================================================================
print("\n[6/8] Extracting TRAIN patches...")

h5_train = h5py.File('X_spatial_train.h5', 'w')
X_spatial_train_dataset = h5_train.create_dataset(
    'X_spatial',
    shape=(train_patch_count, PATCH_SIZE, PATCH_SIZE, 141),
    dtype=np.float32,
    chunks=(min(1000, train_patch_count), PATCH_SIZE, PATCH_SIZE, 141)
)

X_spectral_train = np.zeros((train_patch_count, 141), dtype=np.float32)
Y_train = np.zeros((train_patch_count,), dtype=np.float32)

patch_idx = 0
for apple_id in tqdm(train_apple_ids, desc="  Processing train apples"):
    cube = apples[apple_id].astype(np.float32)
    h, w, bands = cube.shape
    dry_matter = apple_to_dry_matter[apple_id]
    dry_matter_normalized = (dry_matter - dm_min) / (dm_max - dm_min)

    for y in range(HALF_PATCH, h - HALF_PATCH):
        for x in range(HALF_PATCH, w - HALF_PATCH):
            spatial_patch = cube[y-HALF_PATCH:y+HALF_PATCH+1,
                                 x-HALF_PATCH:x+HALF_PATCH+1, :]
            spectral_vector = cube[y, x, :]

            X_spatial_train_dataset[patch_idx] = spatial_patch
            X_spectral_train[patch_idx] = spectral_vector
            Y_train[patch_idx] = dry_matter_normalized

            patch_idx += 1

h5_train.close()
np.save('X_spectral_train.npy', X_spectral_train)
np.save('Y_train.npy', Y_train)

print(f"  ✓ Train: {patch_idx:,} patches")
print(f"  ✓ Y_train range: [{Y_train.min():.4f}, {Y_train.max():.4f}]")
print(f"  ✓ Y_train unique: {len(np.unique(Y_train)):,}")

del X_spectral_train, Y_train
gc.collect()

# ============================================================================
# STEP 7: EXTRACT TEST DATA
# ============================================================================
print("\n[7/8] Extracting TEST patches...")

h5_test = h5py.File('X_spatial_test.h5', 'w')
X_spatial_test_dataset = h5_test.create_dataset(
    'X_spatial',
    shape=(test_patch_count, PATCH_SIZE, PATCH_SIZE, 141),
    dtype=np.float32,
    chunks=(min(1000, test_patch_count), PATCH_SIZE, PATCH_SIZE, 141)
)

X_spectral_test = np.zeros((test_patch_count, 141), dtype=np.float32)
Y_test = np.zeros((test_patch_count,), dtype=np.float32)

patch_idx = 0
for apple_id in tqdm(test_apple_ids, desc="  Processing test apples"):
    cube = apples[apple_id].astype(np.float32)
    h, w, bands = cube.shape
    dry_matter = apple_to_dry_matter[apple_id]
    dry_matter_normalized = (dry_matter - dm_min) / (dm_max - dm_min)

    for y in range(HALF_PATCH, h - HALF_PATCH):
        for x in range(HALF_PATCH, w - HALF_PATCH):
            spatial_patch = cube[y-HALF_PATCH:y+HALF_PATCH+1,
                                 x-HALF_PATCH:x+HALF_PATCH+1, :]
            spectral_vector = cube[y, x, :]

            X_spatial_test_dataset[patch_idx] = spatial_patch
            X_spectral_test[patch_idx] = spectral_vector
            Y_test[patch_idx] = dry_matter_normalized

            patch_idx += 1

h5_test.close()
np.save('X_spectral_test.npy', X_spectral_test)
np.save('Y_test.npy', Y_test)

print(f"  ✓ Test: {patch_idx:,} patches")
print(f"  ✓ Y_test range: [{Y_test.min():.4f}, {Y_test.max():.4f}]")
print(f"  ✓ Y_test unique: {len(np.unique(Y_test)):,}")


📊 Global Normalization Parameters:
  Min: 0.134966
  Max: 0.174347
  Range: 0.039382

[6/8] Extracting TRAIN patches...


  Processing train apples: 100%|██████████| 192/192 [19:44<00:00,  6.17s/it]


  ✓ Train: 465,635 patches
  ✓ Y_train range: [0.0000, 1.0000]
  ✓ Y_train unique: 191

[7/8] Extracting TEST patches...


  Processing test apples: 100%|██████████| 48/48 [05:10<00:00,  6.48s/it]

  ✓ Test: 110,844 patches
  ✓ Y_test range: [0.1284, 0.9033]
  ✓ Y_test unique: 48


In [ ]:
Y_test = np.load('data/Y_test.npy')

print(f"Y_test range: [{Y_test.min():.4f}, {Y_test.max():.4f}]")
print(f"Expected: [0.0, 1.0] or close to it")

# NOW denormalize
csv_data = pd.read_csv('SpectroFood_dataset.csv')
all_dm = csv_data['Dry matter'].values
Y_test_dm = Y_test * (all_dm.max() - all_dm.min()) + all_dm.min()

print(f"\nDenormalized:")
print(f"  Mean: {Y_test_dm.mean():.4f}")
print(f"  Range: [{Y_test_dm.min():.4f}, {Y_test_dm.max():.4f}]")

# Classify
t_low = np.percentile(all_dm, 33.33)
t_high = np.percentile(all_dm, 66.67)

classes = [0 if dm < t_low else 1 if dm < t_high else 2 for dm in Y_test_dm]

print(f"\nClass distribution:")
print(f"  Class 0: {classes.count(0):,} ({classes.count(0)/len(classes)*100:.1f}%)")
print(f"  Class 1: {classes.count(1):,} ({classes.count(1)/len(classes)*100:.1f}%)")
print(f"  Class 2: {classes.count(2):,} ({classes.count(2)/len(classes)*100:.1f}%)")

CSV dry matter range: [0.1350, 0.1743]
Y_test range:         [0.1400, 0.1705]

✅ They're already the same! No denormalization needed!

Class distribution (NO denormalization):
  Class 0: 38,587 (34.8%)
  Class 1: 33,333 (30.1%)
  Class 2: 38,924 (35.1%)


In [14]:
# ============================================================================
# FIXED TRAINING & TESTING DATA ASSESSMENT
# ============================================================================
import numpy as np
import pandas as pd

print("=" * 80)
print("TRAINING & TESTING DATA ASSESSMENT (FIXED)")
print("=" * 80)

# Load both datasets
Y_train = np.load('Y_train.npy')
Y_test = np.load('Y_test.npy')

# Load CSV for reference
csv_data = pd.read_csv('SpectroFood_dataset.csv')
csv_data.columns = csv_data.columns.str.strip()
all_dm = csv_data['Dry matter'].dropna().values
dm_min = all_dm.min()
dm_max = all_dm.max()
dm_range = dm_max - dm_min

print(f"\n📏 Global Dry Matter Stats (from CSV):")
print(f"  Min: {dm_min:.4f}")
print(f"  Max: {dm_max:.4f}")
print(f"  Range: {dm_range:.4f}")

# ============================================================================
# SMART NORMALIZATION DETECTION
# ============================================================================
def detect_normalization(data, expected_min, expected_max, tolerance=0.01):
    """
    Intelligently detect if data is normalized or in original scale.

    Returns:
        is_normalized (bool): True if data appears normalized
        denormalized_data (array): Data in original scale
    """
    data_min, data_max = data.min(), data.max()
    data_range = data_max - data_min

    # Check if data is close to expected raw range
    raw_match = (abs(data_min - expected_min) < tolerance and
                 abs(data_max - expected_max) < tolerance)

    # Check if data is in [0, 1] AND different from raw range
    normalized_match = (data_min >= -0.01 and data_max <= 1.01 and
                       data_range < 1.0 and
                       not raw_match)

    if raw_match:
        # Data is already in original scale
        return False, data
    elif normalized_match:
        # Data is normalized, denormalize it
        denorm = data * (expected_max - expected_min) + expected_min
        return True, denorm
    else:
        # Unclear - default to treating as raw
        print(f"  ⚠️  Warning: Could not confidently determine scale")
        return False, data

# ============================================================================
# ANALYZE TRAINING DATA
# ============================================================================
print("\n" + "=" * 80)
print("📦 TRAINING DATA")
print("=" * 80)

print(f"\n🔍 Raw Y_train values:")
print(f"  Range: [{Y_train.min():.6f}, {Y_train.max():.6f}]")
print(f"  Mean: {Y_train.mean():.6f}")
print(f"  Std: {Y_train.std():.6f}")

train_is_norm, Y_train_dm = detect_normalization(Y_train, dm_min, dm_max)

if train_is_norm:
    print(f"  Status: ✅ NORMALIZED [0,1] → Denormalizing...")
else:
    print(f"  Status: ✅ ALREADY IN ORIGINAL SCALE")

print(f"\n📊 Dry Matter Distribution (Training):")
print(f"  Mean: {Y_train_dm.mean():.6f}")
print(f"  Std: {Y_train_dm.std():.6f}")
print(f"  Range: [{Y_train_dm.min():.6f}, {Y_train_dm.max():.6f}]")

# Verify range matches expected
if abs(Y_train_dm.min() - dm_min) < 0.01 and abs(Y_train_dm.max() - dm_max) < 0.01:
    print(f"  ✅ Range matches global dry matter range")
else:
    print(f"  ⚠️  Range differs from global dry matter range")

# ============================================================================
# CLASSIFICATION ANALYSIS
# ============================================================================
t_low = np.percentile(all_dm, 33.33)
t_high = np.percentile(all_dm, 66.67)

print(f"\n🎯 Classification Thresholds (from CSV):")
print(f"  Unripe:  < {t_low:.6f}")
print(f"  Medium:  {t_low:.6f} - {t_high:.6f}")
print(f"  Ripe:    > {t_high:.6f}")

train_classes = np.where(Y_train_dm < t_low, 0,
                         np.where(Y_train_dm < t_high, 1, 2))

train_class_0 = np.sum(train_classes == 0)
train_class_1 = np.sum(train_classes == 1)
train_class_2 = np.sum(train_classes == 2)

print(f"\n📈 Class Distribution (Training):")
print(f"  Class 0 (Unripe): {train_class_0:,} ({train_class_0/len(train_classes)*100:.1f}%)")
print(f"  Class 1 (Medium): {train_class_1:,} ({train_class_1/len(train_classes)*100:.1f}%)")
print(f"  Class 2 (Ripe):   {train_class_2:,} ({train_class_2/len(train_classes)*100:.1f}%)")

# Calculate imbalance safely
class_counts = [train_class_0, train_class_1, train_class_2]
non_zero_counts = [c for c in class_counts if c > 0]

if len(non_zero_counts) < 2:
    print(f"\n⚠️  WARNING: Only {len(non_zero_counts)} class(es) present!")
    train_imbalance = float('inf')
else:
    max_train = max(non_zero_counts)
    min_train = min(non_zero_counts)
    train_imbalance = max_train / min_train

    print(f"\n⚖️  Imbalance Ratio: {train_imbalance:.2f}:1 ", end="")
    if train_imbalance < 1.5:
        print("✅ BALANCED")
    elif train_imbalance < 3.0:
        print("⚠️  MODERATELY IMBALANCED")
    else:
        print("🚨 SEVERELY IMBALANCED")

# ============================================================================
# ANALYZE TEST DATA
# ============================================================================
print("\n" + "=" * 80)
print("📦 TEST DATA")
print("=" * 80)

print(f"\n🔍 Raw Y_test values:")
print(f"  Range: [{Y_test.min():.6f}, {Y_test.max():.6f}]")
print(f"  Mean: {Y_test.mean():.6f}")
print(f"  Std: {Y_test.std():.6f}")

test_is_norm, Y_test_dm = detect_normalization(Y_test, dm_min, dm_max)

if test_is_norm:
    print(f"  Status: ✅ NORMALIZED [0,1] → Denormalizing...")
else:
    print(f"  Status: ✅ ALREADY IN ORIGINAL SCALE")

print(f"\n📊 Dry Matter Distribution (Test):")
print(f"  Mean: {Y_test_dm.mean():.6f}")
print(f"  Std: {Y_test_dm.std():.6f}")
print(f"  Range: [{Y_test_dm.min():.6f}, {Y_test_dm.max():.6f}]")

if abs(Y_test_dm.min() - Y_train_dm.min()) < 0.02 and abs(Y_test_dm.max() - Y_train_dm.max()) < 0.02:
    print(f"  ✅ Range similar to training data")
else:
    print(f"  ⚠️  Range differs from training data")

test_classes = np.where(Y_test_dm < t_low, 0,
                        np.where(Y_test_dm < t_high, 1, 2))

test_class_0 = np.sum(test_classes == 0)
test_class_1 = np.sum(test_classes == 1)
test_class_2 = np.sum(test_classes == 2)

print(f"\n📈 Class Distribution (Test):")
print(f"  Class 0 (Unripe): {test_class_0:,} ({test_class_0/len(test_classes)*100:.1f}%)")
print(f"  Class 1 (Medium): {test_class_1:,} ({test_class_1/len(test_classes)*100:.1f}%)")
print(f"  Class 2 (Ripe):   {test_class_2:,} ({test_class_2/len(test_classes)*100:.1f}%)")

test_class_counts = [test_class_0, test_class_1, test_class_2]
non_zero_test_counts = [c for c in test_class_counts if c > 0]

if len(non_zero_test_counts) < 2:
    print(f"\n⚠️  WARNING: Only {len(non_zero_test_counts)} class(es) present!")
    test_imbalance = float('inf')
else:
    max_test = max(non_zero_test_counts)
    min_test = min(non_zero_test_counts)
    test_imbalance = max_test / min_test

    print(f"\n⚖️  Imbalance Ratio: {test_imbalance:.2f}:1 ", end="")
    if test_imbalance < 1.5:
        print("✅ BALANCED")
    elif test_imbalance < 3.0:
        print("⚠️  MODERATELY IMBALANCED")
    else:
        print("🚨 SEVERELY IMBALANCED")

# ============================================================================
# COMPARISON & DIAGNOSIS
# ============================================================================
print("\n" + "=" * 80)
print("🔬 DIAGNOSIS")
print("=" * 80)

print(f"\n1️⃣ Scale Consistency Check:")
if train_is_norm == test_is_norm:
    print(f"   ✅ Both datasets have SAME scale")
    print(f"      Training: {'Normalized [0,1]' if train_is_norm else 'Raw dry matter'}")
    print(f"      Testing:  {'Normalized [0,1]' if test_is_norm else 'Raw dry matter'}")
else:
    print(f"   🚨 MISMATCH DETECTED!")
    print(f"      Training: {'Normalized [0,1]' if train_is_norm else 'Raw dry matter'}")
    print(f"      Testing:  {'Normalized [0,1]' if test_is_norm else 'Raw dry matter'}")
    print(f"   ⚠️  This WILL cause model failure!")

print(f"\n2️⃣ Class Presence Check:")
train_present = sum(1 for c in class_counts if c > 0)
test_present = sum(1 for c in test_class_counts if c > 0)
print(f"   Training: {train_present}/3 classes present")
print(f"   Testing:  {test_present}/3 classes present")

if train_present < 3 or test_present < 3:
    print(f"   🚨 CRITICAL: Not all classes present in data!")
    print(f"   👉 This indicates a preprocessing error")

print(f"\n3️⃣ Class Balance Check:")
if train_imbalance != float('inf'):
    print(f"   Training imbalance: {train_imbalance:.2f}:1")
if test_imbalance != float('inf'):
    print(f"   Testing imbalance:  {test_imbalance:.2f}:1")

if train_imbalance > 1.5 and train_imbalance != float('inf'):
    print(f"   🚨 Training data is imbalanced!")
    print(f"   👉 Recommendation: Use class weights or resampling")

# Distribution similarity (only if both have multiple classes)
if train_present >= 2 and test_present >= 2:
    train_dist = np.array(class_counts) / len(train_classes)
    test_dist = np.array(test_class_counts) / len(test_classes)
    dist_diff = np.sum(np.abs(train_dist - test_dist))

    print(f"\n4️⃣ Distribution Similarity:")
    print(f"   Total distribution difference: {dist_diff:.3f}")
    if dist_diff < 0.1:
        print(f"   ✅ Train and test distributions are SIMILAR")
    elif dist_diff < 0.3:
        print(f"   ⚠️  Train and test distributions are SOMEWHAT DIFFERENT")
    else:
        print(f"   🚨 Train and test distributions are VERY DIFFERENT")

# ============================================================================
# FINAL VERDICT
# ============================================================================
print(f"\n" + "=" * 80)
print("🎯 FINAL VERDICT")
print("=" * 80)

issues = []
if train_is_norm != test_is_norm:
    issues.append("Scale mismatch between train/test")
if train_present < 3 or test_present < 3:
    issues.append("Not all classes present (CRITICAL ERROR)")
if train_imbalance > 1.5 and train_imbalance != float('inf'):
    issues.append("Training data is imbalanced")
if train_present >= 2 and test_present >= 2 and dist_diff > 0.3:
    issues.append("Train/test distributions differ significantly")

if not issues:
    print("✅ NO CRITICAL ISSUES DETECTED!")
    print("   Your data looks good to train!")
else:
    print(f"🚨 FOUND {len(issues)} ISSUE(S):")
    for i, issue in enumerate(issues, 1):
        print(f"   {i}. {issue}")
    print("\n👉 Fix these before training!")

print("=" * 80)

TRAINING & TESTING DATA ASSESSMENT (FIXED)

📏 Global Dry Matter Stats (from CSV):
  Min: 0.1350
  Max: 0.1743
  Range: 0.0394

📦 TRAINING DATA

🔍 Raw Y_train values:
  Range: [0.000000, 1.000000]
  Mean: 0.519044
  Std: 0.212527
  ⚠️  Warning: Could not confidently determine scale
  Status: ✅ ALREADY IN ORIGINAL SCALE

📊 Dry Matter Distribution (Training):
  Mean: 0.519044
  Std: 0.212527
  Range: [0.000000, 1.000000]
  ⚠️  Range differs from global dry matter range

🎯 Classification Thresholds (from CSV):
  Unripe:  < 0.151723
  Medium:  0.151723 - 0.159427
  Ripe:    > 0.159427

📈 Class Distribution (Training):
  Class 0 (Unripe): 24,376 (5.2%)
  Class 1 (Medium): 0 (0.0%)
  Class 2 (Ripe):   441,259 (94.8%)

⚖️  Imbalance Ratio: 18.10:1 🚨 SEVERELY IMBALANCED

📦 TEST DATA

🔍 Raw Y_test values:
  Range: [0.128430, 0.903282]
  Mean: 0.517826
  Std: 0.197030
  Status: ✅ NORMALIZED [0,1] → Denormalizing...

📊 Dry Matter Distribution (Test):
  Mean: 0.155359
  Std: 0.007759
  Range: [0.14

In [15]:
# ============================================================================
# STEP 8: FINAL VALIDATION
# ============================================================================
print("\n[8/8] Final Validation Checks...")
print("=" * 80)

Y_train = np.load('Y_train.npy')
Y_test = np.load('Y_test.npy')

# Apple-level statistics
print("\n📊 APPLE-LEVEL STATISTICS:")
print(f"  • Total apples: 240")
print(f"  • Train apples: {len(train_apple_ids)} ({len(train_apple_ids)/240*100:.1f}%)")
print(f"  • Test apples: {len(test_apple_ids)} ({len(test_apple_ids)/240*100:.1f}%)")

# Patch-level statistics
print(f"\n📊 PATCH-LEVEL STATISTICS:")
print(f"  • Train patches: {train_patch_count:,}")
print(f"  • Test patches: {test_patch_count:,}")
print(f"  • Total patches: {train_patch_count + test_patch_count:,}")

# Target distributions
print(f"\n📊 TARGET DISTRIBUTION (Continuous):")
print(f"  Train:")
print(f"    Mean: {Y_train.mean():.6f}")
print(f"    Std:  {Y_train.std():.6f}")
print(f"    Range: [{Y_train.min():.6f}, {Y_train.max():.6f}]")
print(f"\n  Test:")
print(f"    Mean: {Y_test.mean():.6f}")
print(f"    Std:  {Y_test.std():.6f}")
print(f"    Range: [{Y_test.min():.6f}, {Y_test.max():.6f}]")

# Check distribution similarity
mean_diff = abs(Y_train.mean() - Y_test.mean())
std_diff = abs(Y_train.std() - Y_test.std())

print(f"\n📊 DISTRIBUTION SIMILARITY:")
print(f"  • Mean difference: {mean_diff:.6f}")
print(f"  • Std difference: {std_diff:.6f}")
if mean_diff < 0.002 and std_diff < 0.002:
    print(f"  ✓ Distributions are very similar")
else:
    print(f"  ⚠️  Distributions differ slightly")

# Critical leakage check
print(f"\n🔒 LEAKAGE VALIDATION:")
overlap_check = set(train_apple_ids) & set(test_apple_ids)
if overlap_check:
    print(f"  ❌ FAILURE: Apple IDs {overlap_check} appear in both splits!")
    raise ValueError("DATA LEAKAGE DETECTED")
else:
    print(f"  ✅ VERIFIED: No apple appears in both train and test splits")
    print(f"  ✅ VERIFIED: {len(set(train_apple_ids) | set(test_apple_ids))} unique apples total")

# File existence check
print(f"\n📁 OUTPUT FILES:")
expected_files = [
    'X_spatial_train.h5',
    'X_spatial_test.h5',
    'X_spectral_train.npy',
    'X_spectral_test.npy',
    'Y_train.npy',
    'Y_test.npy'
]
for fname in expected_files:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / (1024**2)
        print(f"  ✓ {fname} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ {fname} MISSING!")

print("\n" + "=" * 80)
print("✅ PREPROCESSING COMPLETE - REGRESSION DATASET READY")
print("=" * 80)


[8/8] Final Validation Checks...

📊 APPLE-LEVEL STATISTICS:
  • Total apples: 240
  • Train apples: 192 (80.0%)
  • Test apples: 48 (20.0%)

📊 PATCH-LEVEL STATISTICS:
  • Train patches: 465,635
  • Test patches: 110,844
  • Total patches: 576,479

📊 TARGET DISTRIBUTION (Continuous):
  Train:
    Mean: 0.519044
    Std:  0.212527
    Range: [0.000000, 1.000000]

  Test:
    Mean: 0.517826
    Std:  0.197030
    Range: [0.128430, 0.903282]

📊 DISTRIBUTION SIMILARITY:
  • Mean difference: 0.001218
  • Std difference: 0.015497
  ⚠️  Distributions differ slightly

🔒 LEAKAGE VALIDATION:
  ✅ VERIFIED: No apple appears in both train and test splits
  ✅ VERIFIED: 240 unique apples total

📁 OUTPUT FILES:
  ✓ X_spatial_train.h5 (30328.5 MB)
  ✓ X_spatial_test.h5 (7224.2 MB)
  ✓ X_spectral_train.npy (250.5 MB)
  ✓ X_spectral_test.npy (59.6 MB)
  ✓ Y_train.npy (1.8 MB)
  ✓ Y_test.npy (0.4 MB)

✅ PREPROCESSING COMPLETE - REGRESSION DATASET READY


In [16]:
# ============================================================================
# UPLOAD TO GCS
# ============================================================================
from google.colab import auth
auth.authenticate_user()

GCS_BUCKET = 'processed_data-iyed'
GCS_PREFIX = 'processed_regression_normalizaion'

files_to_upload = [
    'X_spatial_train.h5',
    'X_spatial_test.h5',
    'X_spectral_train.npy',
    'X_spectral_test.npy',
    'Y_train.npy',
    'Y_test.npy'
]

parallel_upload_threshold = '150M'

print(f"\nUploading files to gs://{GCS_BUCKET}/{GCS_PREFIX}/...\n")

for file_path in files_to_upload:
    gsutil_command = (
        f"gsutil -o GSUtil:parallel_composite_upload_threshold={parallel_upload_threshold} cp "
        f"{file_path} gs://{GCS_BUCKET}/{GCS_PREFIX}/"
    )
    print(f"Uploading {file_path}...")
    get_ipython().system(gsutil_command)

print("\n✅ ALL FILES UPLOADED TO GCS SUCCESSFULLY")


Uploading files to gs://processed_data-iyed/processed_regression_normalizaion/...

Uploading X_spatial_train.h5...
Copying file://X_spatial_train.h5 [Content-Type=application/x-hdf5]...
- [1 files][ 29.6 GiB/ 29.6 GiB]   46.2 MiB/s                                   
Operation completed over 1 objects/29.6 GiB.                                     
Uploading X_spatial_test.h5...
Copying file://X_spatial_test.h5 [Content-Type=application/x-hdf5]...
- [1 files][  7.0 GiB/  7.0 GiB]   54.8 MiB/s                                   
Operation completed over 1 objects/7.0 GiB.                                      
Uploading X_spectral_train.npy...
Copying file://X_spectral_train.npy [Content-Type=application/octet-stream]...
| [1 files][250.4 MiB/250.4 MiB]                                                
Operation completed over 1 objects/250.4 MiB.                                    
Uploading X_spectral_test.npy...
Copying file://X_spectral_test.npy [Content-Type=application/octet-stream]...